In [1]:
import torch
import numpy as np
from adapted_sgformer.scripts.train_detection import load_config
from adaptedsgformer.models.detection_models import DetectionGT
from adaptedsgformer.utils import format_data, embed_1D_scalar
from pathlib import Path
from dagr.data.ncaltech101_data import NCaltech101
from torch_geometric.loader import DataLoader
from dagr.data.augment import Augmentations
from argparse import Namespace
from dagr.model.utils import postprocess_network_output, convert_to_training_format
from yolox.models import YOLOX
from torch_scatter import scatter

In [ ]:
TODO:
backbone:
    data = self.events_to_graph(data) --> plus tard

radius: 1.0
uniform_sampling partout

In [2]:
cfg_path = 'config/detection/config_dagt_gnn.yaml'

In [3]:
cfg = load_config(cfg_path)

In [4]:
dataset_path = Path(cfg["data_directory"]) / cfg["dataset"]
augmentations = Augmentations(Namespace(**cfg["augmentations"]))
dataset = NCaltech101(dataset_path, "training", augmentations.transform_training, num_events=cfg["n_nodes"])

In [5]:
len(dataset)

6559

In [6]:
loader = DataLoader(dataset, follow_batch=['bbox', 'bbox0'], shuffle=True, drop_last=True, **cfg["dataloader"])

In [7]:
model = DetectionGT(num_classes=dataset.num_classes, args=cfg["model_params"], height=dataset.height, width=dataset.width).cuda()

poolings: tensor([[0.0179, 0.0250, 1.0000],
        [0.0357, 0.0500, 1.0000],
        [0.0714, 0.1000, 1.0000],
        [0.1429, 0.2000, 1.0000]])
samplings: tensor([2240,  560,  140,   35])


In [8]:
for data in loader:
    data.cuda()
    break

In [10]:
data = format_data(data)
with torch.no_grad():
    outputs = model.forward(data)

ICI


In [13]:
data = format_data(data)
targets = convert_to_training_format(data.bbox, data.bbox_batch, data.num_graphs)

In [14]:
with torch.no_grad():
    fpn_outs = model.backbone(data)

In [14]:
with torch.no_grad():
    loss, iou_loss, conf_loss, cls_loss, l1_loss, num_fg = model.head(
        fpn_outs, targets, data
    )

In [15]:
loss, iou_loss, conf_loss, cls_loss, l1_loss, num_fg

(tensor(100.7543, device='cuda:0'),
 tensor(4.9840, device='cuda:0'),
 tensor(20.8060, device='cuda:0'),
 tensor(74.9642, device='cuda:0'),
 0.0,
 1.0)

In [13]:
fpn_outs

DataBatch(x=[280, 64], pos=[280, 3], width=[8], height=[8], time_window=[8], dist_mat=[280, 35], batch=[280], ptr=[9], edge_index=[2, 3240], edge_attr=[3240, 3])

In [15]:
self = model.head

In [16]:
xin = fpn_outs.clone()

In [17]:
        ev_out = dict(outputs=[], origin_preds=[], x_shifts=[], y_shifts=[], expanded_strides=[])

        batch_size = xin.num_graphs
        normalizer = torch.stack([xin.width[0], xin.height[0]], dim=0)

        #Process data for the first stage of detection 
        cls_output, reg_output, obj_output = self.process_feature(xin)

        self.collect_outputs(cls_output, reg_output, obj_output, self.strides[0], batch_size, normalizer, ret=ev_out)

In [36]:
from adaptedsgformer.layers.heads import GNNHead, SparseYoloxHead

from argparse import Namespace

In [39]:
        head_args = dict(
            num_classes=dataset.num_classes,
            strides=model.backbone.strides,
            in_channels=model.backbone.hidden_channels_list[-model.backbone.num_scales:], 
            args=Namespace(**cfg["model_params"]['head'])
        )
        head = GNNHead(**head_args).cuda()

In [41]:
model.backbone.poolings[-1]

tensor([0.1429, 0.2000, 1.0000])

In [44]:
xin = fpn_outs.clone()
xin.pooling = model.backbone.poolings[-1].cuda()
batch_size = xin.num_graphs
cls_output_orig, reg_output_orig, obj_output_orig = head.process_feature(xin, head.stem1, head.cls_conv1, head.reg_conv1,
                                                        head.cls_pred1, head.reg_pred1, head.obj_pred1, batch_size=batch_size, cache=head.cache)

In [46]:
output_orig = torch.cat([reg_output_orig, obj_output_orig, cls_output_orig], 1)

In [47]:
output_orig.shape

torch.Size([8, 105, 5, 7])

In [45]:
cls_output_orig.shape, reg_output_orig.shape, obj_output_orig.shape

(torch.Size([8, 100, 5, 7]),
 torch.Size([8, 4, 5, 7]),
 torch.Size([8, 1, 5, 7]))

In [ ]:
TODO (training):

- anchors (locations) are no longer pixel locations but node positions:
    -> modify get_output_and_grid & get_geometry_constraint (what stride to choose?)
    -> x_shifts and y_shift become node x and node y

- implem -> how to code unif sampling (esp. accumulation of dropped nodes features)

- solve problems w/ current implem -> pe max period & dim, graph & edge creation, edge attributes

(inference)

In [18]:
outputs = torch.cat(ev_out['outputs'], 1)

In [19]:
outputs.shape

torch.Size([8, 35, 105])

In [22]:
labels = targets

In [23]:
x_shifts, y_shifts, expanded_strides, origin_preds = ev_out['x_shifts'], ev_out['y_shifts'], ev_out['expanded_strides'], \
    ev_out['origin_preds']

In [24]:
labels.shape

torch.Size([8, 100, 5])

In [25]:
        bbox_preds = outputs[:, :, :4]  # [batch, n_anchors_all, 4]
        obj_preds = outputs[:, :, 4:5]  # [batch, n_anchors_all, 1]
        cls_preds = outputs[:, :, 5:]  # [batch, n_anchors_all, n_cls]

        # calculate targets
        nlabel = (labels.sum(dim=2) > 0).sum(dim=1)  # number of objects

        total_num_anchors = outputs.shape[1]
        x_shifts = torch.cat(x_shifts, 1)  # [1, n_anchors_all]
        y_shifts = torch.cat(y_shifts, 1)  # [1, n_anchors_all]
        expanded_strides = torch.cat(expanded_strides, 1)

In [27]:
        cls_targets = []
        reg_targets = []
        l1_targets = []
        obj_targets = []
        fg_masks = []

        num_fg = 0.0
        num_gts = 0.0

        for batch_idx in range(outputs.shape[0]):
            num_gt = int(nlabel[batch_idx])
            num_gts += num_gt
            break

In [28]:
                gt_bboxes_per_image = labels[batch_idx, :num_gt, 1:5]
                gt_classes = labels[batch_idx, :num_gt, 0]
                bboxes_preds_per_image = bbox_preds[batch_idx]

In [29]:
gt_bboxes_per_image

tensor([[ 92.0000,  52.5000, 172.0000,  79.0000]], device='cuda:0')

In [ ]:
(gt_matched_classes,
    fg_mask,
    pred_ious_this_matching,
    matched_gt_inds,
    num_fg_img,
) = self.get_assignments( 

In [84]:
        fg_mask, geometry_relation = self.get_geometry_constraint(
            batch_idx,
            gt_bboxes_per_image,
            expanded_strides,
            x_shifts,
            y_shifts,
        )

In [85]:
fg_mask

tensor([False, False,  True,  True, False, False, False, False,  True,  True,
         True,  True,  True,  True,  True,  True, False, False,  True,  True,
         True,  True, False, False, False,  True,  True, False, False, False,
        False, False, False, False,  True], device='cuda:0')

In [40]:
while not fg_mask.any():
    print('L0')

In [29]:
fg_mask

tensor([ True,  True,  True,  True,  True,  True, False,  True, False, False,
        False,  True,  True, False,  True, False,  True,  True,  True, False,
         True, False, False,  True,  True,  True,  True,  True, False, False,
        False, False,  True, False,  True], device='cuda:0')

In [33]:
        bboxes_preds_per_image = bboxes_preds_per_image[fg_mask]
        cls_preds_ = cls_preds[batch_idx][fg_mask]
        obj_preds_ = obj_preds[batch_idx][fg_mask]
        num_in_boxes_anchor = bboxes_preds_per_image.shape[0]

In [32]:
import torch.nn.functional as F
from yolox.utils import bboxes_iou

In [35]:
gt_bboxes_per_image.shape, bboxes_preds_per_image.shape

(torch.Size([1, 4]), torch.Size([17, 4]))

In [34]:
pair_wise_ious = bboxes_iou(gt_bboxes_per_image, bboxes_preds_per_image, False)

In [36]:
pair_wise_ious.shape

torch.Size([1, 17])

In [37]:
pair_wise_ious

tensor([[0.0000, -0.0000, 0.0000, 0.0004, -0.0000, 0.0154, -0.0000, 0.0000, 0.0889,
         0.0700, -0.0000, -0.0000, 0.0000, -0.0000, 0.0899, -0.0000, -0.0000]],
       device='cuda:0', grad_fn=<DivBackward0>)

In [41]:
        gt_cls_per_image = (
            F.one_hot(gt_classes.to(torch.int64), self.num_classes)
            .float()
        )
        pair_wise_ious_loss = -torch.log(pair_wise_ious + 1e-8)

In [42]:
pair_wise_ious_loss

tensor([[18.4207, 18.4207, 18.4207,  7.9092, 18.4207,  4.1738, 18.4207, 18.4207,
          2.4201,  2.6591, 18.4207, 18.4207, 18.4207, 18.4207,  2.4093, 18.4207,
         18.4207]], device='cuda:0', grad_fn=<NegBackward0>)

In [43]:
        with torch.amp.autocast('cuda', enabled=False):
            cls_preds_ = (
                cls_preds_.float().sigmoid_() * obj_preds_.float().sigmoid_()
            ).sqrt()
            pair_wise_cls_loss = F.binary_cross_entropy(
                cls_preds_.unsqueeze(0).repeat(num_gt, 1, 1),
                gt_cls_per_image.unsqueeze(1).repeat(1, num_in_boxes_anchor, 1),
                reduction="none"
            ).sum(-1)
        del cls_preds_

In [44]:
pair_wise_cls_loss

tensor([[75.2810, 64.7900, 83.6431, 60.2729, 55.1189, 62.1280, 59.9255, 63.7199,
         68.5502, 59.3104, 71.8667, 77.8036, 50.9072, 74.2916, 73.1938, 66.0494,
         81.0111]], device='cuda:0', grad_fn=<SumBackward1>)

In [47]:
        cost = (
            pair_wise_cls_loss
            + 3.0 * pair_wise_ious_loss
            + float(1e6) * (~geometry_relation)
        )

In [48]:
cost

tensor([[130.5431, 120.0520, 138.9052,  84.0006, 110.3810,  74.6496, 115.1876,
         118.9819,  75.8104,  67.2878, 127.1288, 133.0657, 106.1693, 129.5537,
          80.4217, 121.3115, 136.2731]], device='cuda:0',
       grad_fn=<AddBackward0>)

In [45]:
gt_classes, num_gt, fg_mask

(tensor([4.], device='cuda:0'),
 1,
 tensor([False, False,  True,  True, False, False, False, False,  True,  True,
          True,  True,  True,  True,  True,  True, False, False,  True,  True,
          True,  True, False, False, False,  True,  True, False, False, False,
         False, False, False, False,  True], device='cuda:0'))

In [49]:
        (
            num_fg,
            gt_matched_classes,
            pred_ious_this_matching,
            matched_gt_inds,
        ) = self.simota_matching(cost, pair_wise_ious, gt_classes, num_gt, fg_mask)

In [96]:
        matching_matrix = torch.zeros_like(cost, dtype=torch.uint8)

        n_candidate_k = min(10, pair_wise_ious.size(1))
        topk_ious, _ = torch.topk(pair_wise_ious, n_candidate_k, dim=1)
        dynamic_ks = torch.clamp((20 * topk_ious.sum(1)).int(), min=1)

In [101]:
dynamic_ks * 0 + n_candidate_k

tensor([10], device='cuda:0', dtype=torch.int32)

In [102]:
n_candidate_k

10

In [98]:
topk_ious

tensor([[0.0899, 0.0889, 0.0700, 0.0154, 0.0004, 0.0000, -0.0000, 0.0000, 0.0000,
         0.0000]], device='cuda:0', grad_fn=<TopkBackward0>)

In [ ]:
(20 * topk_ious.sum(1)).int()

In [97]:
dynamic_ks

tensor([5], device='cuda:0', dtype=torch.int32)

In [63]:
        for gt_idx in range(num_gt):
            _, pos_idx = torch.topk(
                cost[gt_idx], k=dynamic_ks[gt_idx], largest=False
            )
            matching_matrix[gt_idx][pos_idx] = 1

In [58]:
cost

tensor([[130.5431, 120.0520, 138.9052,  84.0006, 110.3810,  74.6496, 115.1876,
         118.9819,  75.8104,  67.2878, 127.1288, 133.0657, 106.1693, 129.5537,
          80.4217, 121.3115, 136.2731]], device='cuda:0',
       grad_fn=<AddBackward0>)

In [87]:
matching_matrix

tensor([[0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0]], device='cuda:0',
       dtype=torch.uint8)

In [65]:
        anchor_matching_gt = matching_matrix.sum(0)
        # deal with the case that one anchor matches multiple ground-truths
        if anchor_matching_gt.max() > 1:
            multiple_match_mask = anchor_matching_gt > 1
            _, cost_argmin = torch.min(cost[:, multiple_match_mask], dim=0)
            matching_matrix[:, multiple_match_mask] *= 0
            matching_matrix[cost_argmin, multiple_match_mask] = 1
        fg_mask_inboxes = anchor_matching_gt > 0
        num_fg = fg_mask_inboxes.sum().item()

In [66]:
num_fg

5

In [75]:
x, y = x_shifts[batch_idx] * expanded_strides[0][0], y_shifts[batch_idx] * expanded_strides[0][1]

In [79]:
x.shape

torch.Size([35])

In [80]:
fg_mask.shape

torch.Size([35])

In [82]:
fg_mask

tensor([False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False,  True, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False], device='cuda:0')

In [88]:
matching_matrix.shape

torch.Size([1, 17])

In [93]:
x[fg_mask.nonzero()][:, 0][matching_matrix[0]]

/tmp/ipykernel_37307/2888822417.py:1: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:29.)
  x[fg_mask.nonzero()][:, 0][matching_matrix[0]]


tensor([51., 50., 53., 54., 66.], device='cuda:0')

In [95]:
x[fg_mask.nonzero()][:, 0]

tensor([118.,  72., 124.,  51.,  85.,  50., 107., 120.,  53.,  54.,  76.,  53.,
         44.,  48.,  66.,  74.,  61.], device='cuda:0')

In [94]:
y[fg_mask.nonzero()][:, 0][matching_matrix[0]]

/tmp/ipykernel_37307/3736721846.py:1: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at /pytorch/aten/src/ATen/native/IndexingUtils.h:29.)
  y[fg_mask.nonzero()][:, 0][matching_matrix[0]]


tensor([58., 45., 33., 61., 55.], device='cuda:0')

In [73]:
x_shifts[batch_idx] * expanded_strides[0][0]

tensor([ 22.,  24., 118.,  72., 166.,  65., 187.,  28., 124.,  51.,  85.,  50.,
        107., 120.,  53.,  54., 161.,   6.,  76.,  53.,  44.,  48.,  20., 191.,
          3.,  66.,  74., 165., 163.,  35.,  34., 169., 172.,  29.,  61.],
       device='cuda:0')

In [74]:
y_shifts[batch_idx] * expanded_strides[0][1]

tensor([ 89.0000,  68.0000,  25.0000,  59.0000,  87.0000, 115.0000,  53.0000,
         24.0000,  24.0000,  58.0000,  80.0000,  45.0000,  79.0000,  62.0000,
         33.0000,  61.0000,  48.0000,  94.0000,  37.0000,  31.0000, 100.0000,
         41.0000,  53.0000,  46.0000,  27.0000,  55.0000,  96.0000,  27.0000,
         26.0000,  83.0000,  39.0000,  40.0000,  55.0000,  32.0000,  44.0000],
       device='cuda:0')

In [40]:
with torch.no_grad():
    model_outputs = model(data)

In [41]:
model_outputs

{'total_loss': tensor(104.3970, device='cuda:0'),
 'iou_loss': tensor(4.9290, device='cuda:0'),
 'l1_loss': 0.0,
 'conf_loss': tensor(25.2903, device='cuda:0'),
 'cls_loss': tensor(74.1776, device='cuda:0'),
 'num_fg': 1.0}